# AlpinaSignal - GPU Training
## Train all 45 models on free Google Colab GPU

**Steps:**
1. Check GPU is enabled
2. Clone repo from GitHub
3. Upload data_cache.zip
4. Install dependencies
5. Train all models
6. Download trained models

## Step 1: Check GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("NO GPU! Go to Runtime -> Change runtime type -> GPU")

## Step 2: Clone repo

In [ ]:
!git clone https://github.com/alpinasignal/alpinas.git
%cd alpinas

## Step 3: Upload data_cache.zip
Upload the `data_cache.zip` file when prompted

In [ ]:
from google.colab import files
import zipfile
import os

print("Upload data_cache.zip...")
uploaded = files.upload()

# Extract - make sure we're in the repo directory
for fn in uploaded.keys():
    if fn.endswith('.zip'):
        with zipfile.ZipFile(fn, 'r') as z:
            # Check if zip contains data_cache/ prefix
            first = z.namelist()[0]
            if first.startswith('data_cache/'):
                z.extractall('.')
            else:
                # Files without prefix - extract into data_cache/
                os.makedirs('data_cache', exist_ok=True)
                z.extractall('data_cache')
            print(f"Extracted {fn}")

# Verify
if not os.path.exists('data_cache'):
    # Maybe extracted one level up, try to find it
    for d in ['../data_cache', '/content/data_cache']:
        if os.path.exists(d):
            os.system(f'cp -r {d} .')
            print(f"Copied data_cache from {d}")
            break

csv_files = [f for f in os.listdir('data_cache') if f.endswith('.csv')]
print(f"\nFound {len(csv_files)} CSV files in data_cache/")
for f in sorted(csv_files)[:5]:
    print(f"  {f}")
if len(csv_files) > 5:
    print(f"  ... and {len(csv_files)-5} more")

## Step 4: Install dependencies

In [ ]:
!pip install -q xgboost lightgbm optuna ccxt ta loguru python-telegram-bot fastapi uvicorn pydantic python-dotenv sqlalchemy aiosqlite psycopg2-binary tqdm

## Step 5: Train ALL 45 models on GPU
This takes ~1-2 hours on T4 GPU (vs 30+ hours on CPU)

In [ ]:
!python train_all.py

## Step 6: Download trained models

In [ ]:
import zipfile
import os
from google.colab import files

# List all trained models
model_files = [f for f in os.listdir('models') if f.endswith('.pt') or f.endswith('.pkl')]
print(f"Trained models: {len(model_files)}")
for f in sorted(model_files):
    size_mb = os.path.getsize(os.path.join('models', f)) / 1024 / 1024
    print(f"  {f} ({size_mb:.1f} MB)")

# Zip all models
with zipfile.ZipFile('trained_models.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in model_files:
        zf.write(os.path.join('models', f), f'models/{f}')

total_size = os.path.getsize('trained_models.zip') / 1024 / 1024
print(f"\nArchive: trained_models.zip ({total_size:.1f} MB)")

# Download
files.download('trained_models.zip')
print("Download started!")